## House Prices (Data Quality)

In [1]:
import pandas as pd
from IPython.display import display

from house_prices_mlops.pipeline import load_stage
from house_prices_mlops.data_tracker import DataTracker

df_hp = load_stage("train")
tracker = DataTracker.load(df_hp)


2026-10-11 06:14:23.555 | INFO     | house_prices_mlops.config:<module>:11 - PROJ_ROOT path is: /home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops


Only training data is reviewed. These analyses identify quality issues and potential treatments; they do not modify the data or apply imputation.

### Quality Profile

In [2]:
display(tracker.profile)


,var,data_type,variable_type,unique_count,missing_count,missing_pct,zero_count,zero_pct,min,max,mean,median,std,skewness,kurtosis
0,Id,int64,identifier,1168,0,0.000000,0.0,0.0,1.0,1460.0,730.904966,732.5,425.369088,0.000920,-1.215850
1,MSSubClass,int64,nominal,15,0,0.000000,0.0,0.0,20.0,190.0,56.849315,50.0,42.531862,1.438804,1.698354
2,MSZoning,object,nominal,5,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,LotFrontage,float64,continuous,107,217,18.578767,0.0,0.0,21.0,313.0,70.343849,70.0,24.897021,2.406787,19.454319
4,LotArea,int64,continuous,890,0,0.000000,0.0,0.0,1300.0,215245.0,10689.642123,9600.0,10759.366198,11.958088,186.444082
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,MoSold,int64,temporal,12,0,0.000000,0.0,0.0,1.0,12.0,6.356164,6.0,2.670707,0.241761,-0.371218
77,YrSold,int64,temporal,5,0,0.000000,0.0,0.0,2006.0,2010.0,2007.818493,2008.0,1.322639,0.082593,-1.187097
78,SaleType,object,nominal,9,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,SaleCondition,object,nominal,6,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Missing Values

In [3]:
missing_review = tracker.profile[
    tracker.profile['missing_count'] > 0
][
    [
        'var',
        'variable_type',
        'missing_count',
        'missing_pct'
    ]
].sort_values(
    'missing_pct',
    ascending=False
)

display(missing_review)

,var,variable_type,missing_count,missing_pct
72,PoolQC,ordinal,1162,99.486301
74,MiscFeature,nominal,1122,96.061644
6,Alley,nominal,1094,93.664384
73,Fence,ordinal,935,80.051370
25,MasVnrType,nominal,683,58.476027
57,FireplaceQu,ordinal,547,46.832192
3,LotFrontage,continuous,217,18.578767
58,GarageType,nominal,64,5.479452
59,GarageYrBlt,temporal,64,5.479452
60,GarageFinish,ordinal,64,5.479452


#### Missing Values Nature

In [ ]:
missing_analysis = tracker.profile[
    tracker.profile['missing_count'] > 0
][
    [
        'var',
        'variable_type',
        'missing_count',
        'missing_pct'
    ]
].copy()

missing_analysis['missing_nature'] = None
missing_analysis['notes'] = None

structural_missing = [
    'PoolQC',
    'MiscFeature',
    'Alley',
    'Fence',
    'FireplaceQu',
    'GarageType',
    'GarageYrBlt',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'BsmtQual',
    'BsmtCond'
]

tracker.add_finding(key='structural_missing_features', 
    var='structural_missing_features',
    stage='DU - Quality - Missing Values',
    finding=(
        "Se identificaron valores faltantes estructurales en las variables: "
        "PoolQC, MiscFeature, Alley, Fence, FireplaceQu, "
        "GarageType, GarageYrBlt, GarageFinish, GarageQual, "
        "GarageCond, BsmtExposure, BsmtFinType1, BsmtFinType2, "
        "BsmtQual y BsmtCond. Estos valores faltantes representan "
        "la ausencia de la característica correspondiente en la vivienda."
    )
)


missing_analysis.loc[
    missing_analysis['var'].isin(structural_missing),
    'missing_nature'
] = 'structural'

display(missing_analysis.loc[missing_analysis['missing_nature'].isnull()].sort_values('missing_pct',ascending=False))

#### LotFrontage

In [ ]:
# LotFrontage

lotfrontage_by_neighborhood = (
    df_hp
    .groupby('Neighborhood')['LotFrontage']
    .agg(['count', 'median', 'mean'])
    .sort_values('median')
)
missing_by_neighborhood = (
    df_hp
    .groupby('Neighborhood')['LotFrontage']
    .agg(
        total='size',
        missing=lambda x: x.isna().sum()
    )
)

missing_by_neighborhood['missing_pct'] = (
    missing_by_neighborhood['missing']
    / missing_by_neighborhood['total']
    * 100
)

display(
    missing_by_neighborhood.sort_values(
        'missing_pct',
        ascending=False
    )
)
display(lotfrontage_by_neighborhood)

tracker.add_finding(key='lot_frontage_missing', 
    var='LotFrontage',
    stage='DU - Quality - Missing Values',
    finding=(
        f"{df_hp['LotFrontage'].isna().mean() * 100:.2f}% de valores faltantes en entrenamiento. La distribución de LotFrontage varía según "
        'Neighborhood; candidata a imputación por Neighborhood.'
    )
)


#### MasVnrType and MasVnrArea

In [6]:
# Masonry veneer
masvnr_review = (
    df_hp.loc[
        df_hp['MasVnrType'].isna() | df_hp['MasVnrArea'].isna(),
        ['MasVnrType', 'MasVnrArea']
    ]
    .value_counts(dropna=False)
    .reset_index(name='count')
)

display(masvnr_review)

,MasVnrType,MasVnrArea,count
0,NaN,0.0,672
1,NaN,NaN,6
2,NaN,1.0,2
3,NaN,288.0,1
4,NaN,312.0,1
5,NaN,344.0,1


In [ ]:
no_veneer = df_hp['MasVnrType'].isna() & df_hp['MasVnrArea'].eq(0)
unknown_type = df_hp['MasVnrType'].isna() & df_hp['MasVnrArea'].gt(0)
unknown_both = df_hp['MasVnrType'].isna() & df_hp['MasVnrArea'].isna()
tracker.add_finding(key='mas_vnr_type_structural_absence', var='MasVnrType', stage='DU - Quality - Missing Values',
    finding=f'{no_veneer.sum()} registros de entrenamiento presentan MasVnrType faltante y MasVnrArea igual a cero; probable ausencia estructural.')
tracker.add_finding(key='mas_vnr_type_positive_area_missing', var='MasVnrType', stage='DU - Quality - Inconsistency',
    finding=f'{unknown_type.sum()} registros de entrenamiento presentan MasVnrType faltante pese a tener MasVnrArea positiva.')
tracker.add_finding(key='mas_vnr_area_missing', var='MasVnrArea', stage='DU - Quality - Missing Values',
    finding=f'{unknown_both.sum()} registros de entrenamiento presentan tanto MasVnrType como MasVnrArea faltantes; requieren una decisión de tratamiento.')


#### Electrical

In [8]:
display(
    df_hp.loc[
        df_hp['Electrical'].isna(),
        ['Electrical', 'YearBuilt', 'Neighborhood', 'OverallQual']
    ]
)

display(
    df_hp['Electrical'].value_counts(dropna=False)
)

,Electrical,YearBuilt,Neighborhood,OverallQual
1379,NaN,2006,Timber,5


Electrical
SBrkr    1071
FuseA      69
FuseF      24
FuseP       3
NaN         1
Name: count, dtype: int64

In [ ]:
electrical_counts = df_hp['Electrical'].value_counts()
tracker.add_finding(key='electrical_missing', var='Electrical', stage='DU - Quality - Missing Values',
    finding=(f"{df_hp['Electrical'].isna().sum()} valores faltantes "
             f"({df_hp['Electrical'].isna().mean() * 100:.2f}%) en entrenamiento. "
             f"Categoría dominante: {electrical_counts.index[0]} "
             f"({electrical_counts.iloc[0]}/{len(df_hp)}); revisar la imputación en DP."))


### Duplicates

In [10]:
#Duplicates
identifier_cols = tracker.profile.loc[
    tracker.profile['variable_type'] == 'identifier',
    'var'
].tolist()

duplicate_without_id = df_hp.duplicated(
    subset=[
        col for col in df_hp.columns
        if col not in identifier_cols
    ]
)

print(
    'Duplicates excluding identifiers:',
    duplicate_without_id.sum()
)

Duplicates excluding identifiers: 0


### Domain Constraints

#### Continuous Variables

In [11]:

# DOMAIN CONSTRAINTS - CONTINUOUS

continuous_review = tracker.profile.loc[
    tracker.profile['variable_type'] == 'continuous',
    [
        'var',
        'min',
        'max'
    ]
].copy()

continuous_review['has_negative_values'] = (
    continuous_review['min'] < 0
)

display(continuous_review)

,var,min,max,has_negative_values
3,LotFrontage,21.0,313.0,False
4,LotArea,1300.0,215245.0,False
26,MasVnrArea,0.0,1378.0,False
34,BsmtFinSF1,0.0,5644.0,False
36,BsmtFinSF2,0.0,1127.0,False
37,BsmtUnfSF,0.0,2336.0,False
38,TotalBsmtSF,0.0,6110.0,False
43,1stFlrSF,334.0,4692.0,False
44,2ndFlrSF,0.0,2065.0,False
45,LowQualFinSF,0.0,572.0,False


#### Discrete Variables / Counts

In [12]:
#Domain Contraints
# DOMAIN CONSTRAINTS - DISCRETE / COUNTS

count_review = tracker.profile.loc[
    tracker.profile['variable_type'] == 'discrete',
    [
        'var',
        'min',
        'max'
    ]
].copy()

count_review['has_negative_values'] = (
    count_review['min'] < 0
)

display(count_review)

,var,min,max,has_negative_values
47,BsmtFullBath,0.0,3.0,False
48,BsmtHalfBath,0.0,2.0,False
49,FullBath,0.0,3.0,False
50,HalfBath,0.0,2.0,False
51,BedroomAbvGr,0.0,8.0,False
52,KitchenAbvGr,0.0,3.0,False
54,TotRmsAbvGrd,2.0,14.0,False
56,Fireplaces,0.0,3.0,False
61,GarageCars,0.0,4.0,False


#### Ordinal Variables (1–10)

In [13]:
# DOMAIN CONSTRAINTS - ORDINAL 1-10

ordinal_1_10 = [
    'OverallQual',
    'OverallCond'
]

ordinal_review = tracker.profile.loc[
    tracker.profile['var'].isin(ordinal_1_10),
    ['var', 'min', 'max']
].copy()

ordinal_review['has_out_of_range_values'] = (
    (ordinal_review['min'] < 1) |
    (ordinal_review['max'] > 10)
)

display(ordinal_review)

,var,min,max,has_out_of_range_values
17,OverallQual,1.0,10.0,False
18,OverallCond,1.0,9.0,False


### Inconsistency

#### Temporal Relationships

In [14]:

# INCONSISTENCY - TEMPORAL RELATIONSHIPS


time_variables = [
    'Id',
    'YearBuilt',
    'YearRemodAdd',
    'GarageYrBlt',
    'YrSold'
]

df_validation = df_hp[time_variables].copy()

# Remodelación anterior a construcción
df_validation['remodel_before_build'] = (
    df_validation['YearRemodAdd'] < df_validation['YearBuilt']
)

# Venta anterior a construcción
df_validation['sale_before_build'] = (
    df_validation['YrSold'] < df_validation['YearBuilt']
)

# Garage anterior a construcción
df_validation['garage_before_build'] = (
    df_validation['GarageYrBlt'] < df_validation['YearBuilt']
)

invalid_rows = df_validation[
    df_validation[
        [
            'remodel_before_build',
            'sale_before_build',
            'garage_before_build'
        ]
    ].any(axis=1)
]

display(invalid_rows)

,Id,YearBuilt,YearRemodAdd,GarageYrBlt,YrSold,remodel_before_build,sale_before_build,garage_before_build
93,94,1910,1998,1900.0,2007,False,False,True
600,601,2005,2005,2003.0,2006,False,False,True
1103,1104,1959,1959,1954.0,2006,False,False,True
1376,1377,1930,1950,1925.0,2008,False,False,True
1418,1419,1963,1963,1962.0,2008,False,False,True


In [ ]:
garage_before_build = df_validation['garage_before_build'].sum()
tracker.add_finding(key='garage_year_before_house', var='GarageYrBlt', stage='DU - Quality - Inconsistency',
    finding=f'{garage_before_build} registros de entrenamiento presentan GarageYrBlt anterior a YearBuilt; revisar en DP.')


### Constant / Near-Constant Variables

#### Including Missing Values

In [16]:

# CONSTANT / NEAR-CONSTANT VARIABLES


categorical_vars = tracker.profile.loc[
    tracker.profile['variable_type'].isin(['nominal', 'ordinal']),
    'var'
].tolist()

high_freq = []

for col in categorical_vars:

    freq = df_hp[col].value_counts(
        normalize=True,
        dropna=False
    )

    high_freq.append({
        'var': col,
        'top_value': freq.index[0],
        'top_frequency_pct': freq.iloc[0] * 100
    })

high_freq_df = pd.DataFrame(high_freq)

display(
    high_freq_df.sort_values(
        'top_frequency_pct',
        ascending=False
    )
)

,var,top_value,top_frequency_pct
6,Utilities,AllPub,99.914384
2,Street,Pave,99.657534
41,PoolQC,NaN,99.486301
11,Condition2,Norm,99.058219
17,RoofMatl,CompShg,98.373288
29,Heating,GasA,97.602740
43,MiscFeature,NaN,96.061644
8,LandSlope,Gtl,94.863014
3,Alley,NaN,93.664384
31,CentralAir,Y,92.893836


#### Non-Missing Values

In [17]:

# NEAR-CONSTANT - NON-MISSING VALUES


high_freq = []

for col in categorical_vars:

    freq = df_hp[col].value_counts(
        normalize=True,
        dropna=True
    )

    if len(freq) > 0:
        high_freq.append({
            'var': col,
            'top_value': freq.index[0],
            'top_frequency_pct': freq.iloc[0] * 100
        })

high_freq_df = pd.DataFrame(high_freq)

near_constant = high_freq_df[
    high_freq_df['top_frequency_pct'] >= 95
].sort_values(
    'top_frequency_pct',
    ascending=False
)

display(near_constant)

,var,top_value,top_frequency_pct
6,Utilities,AllPub,99.914384
2,Street,Pave,99.657534
11,Condition2,Norm,99.058219
17,RoofMatl,CompShg,98.373288
29,Heating,GasA,97.602740
39,GarageCond,TA,96.105072
38,GarageQual,TA,95.108696


In [ ]:
near_constant_keys = {'Utilities': 'utilities_near_constant', 'Street': 'street_near_constant', 'Condition2': 'condition2_near_constant', 'RoofMatl': 'roof_material_near_constant', 'Heating': 'heating_near_constant', 'GarageCond': 'garage_condition_near_constant', 'GarageQual': 'garage_quality_near_constant'}

for _, row in near_constant.iterrows():

    tracker.add_finding(key=near_constant_keys[row["var"]], 
        var=row['var'],
        stage='DU - Quality - Near Constant',
        finding=(
            f"La categoría dominante '{row['top_value']}' representa el "
            f"{row['top_frequency_pct']:.2f}% de las observaciones sin valores faltantes; "
            "requiere revisar su relevancia."
        )
    )

### Save Findings

In [19]:
tracker.save()
